In [2]:
import pandas as pd

column_names = [
    "engine_id",
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
    *[f"sensor_{i}" for i in range(1,22)]
]

train_df = pd.read_csv(
    "../data/train_FD001.txt",
    sep=r"\s+",
    header=None
)

train_df = train_df.iloc[:, :26]

train_df.columns = column_names

In [3]:
train_df.shape

(20631, 26)

In [4]:
train_df.head()

,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [5]:
train_df["RUL"] = (
    train_df.groupby("engine_id")["cycle"]
    .transform("max")
    - train_df["cycle"]
)

train_df[["engine_id", "cycle", "RUL"]].head(10)

,engine_id,cycle,RUL
0,1,1,191
1,1,2,190
2,1,3,189
3,1,4,188
4,1,5,187
5,1,6,186
6,1,7,185
7,1,8,184
8,1,9,183
9,1,10,182


In [6]:
#Create rolling average features
important_sensors = [
    "sensor_2",
    "sensor_4",
    "sensor_11",
    "sensor_15",
    "sensor_17"
]

for sensor in important_sensors:

    train_df[f"{sensor}_avg5"] = (
        train_df.groupby("engine_id")[sensor]
        .transform(lambda x: x.rolling(5, min_periods=1).mean())
    )

    train_df[f"{sensor}_avg10"] = (
        train_df.groupby("engine_id")[sensor]
        .transform(lambda x: x.rolling(10, min_periods=1).mean())
    )

    train_df[f"{sensor}_avg20"] = (
        train_df.groupby("engine_id")[sensor]
        .transform(lambda x: x.rolling(20, min_periods=1).mean())
    )

    train_df[f"{sensor}_avg30"] = (
        train_df.groupby("engine_id")[sensor]
        .transform(lambda x: x.rolling(30, min_periods=1).mean())
    )

In [7]:
#Create difference features
for sensor in important_sensors:

    train_df[f"{sensor}_diff"] = (
        train_df.groupby("engine_id")[sensor]
        .diff()
        .fillna(0)
    )

In [8]:
train_df.shape

(20631, 52)

In [9]:
len(train_df.columns)

52

In [10]:
train_df.columns.tolist()

['engine_id',
 'cycle',
 'op_setting_1',
 'op_setting_2',
 'op_setting_3',
 'sensor_1',
 'sensor_2',
 'sensor_3',
 'sensor_4',
 'sensor_5',
 'sensor_6',
 'sensor_7',
 'sensor_8',
 'sensor_9',
 'sensor_10',
 'sensor_11',
 'sensor_12',
 'sensor_13',
 'sensor_14',
 'sensor_15',
 'sensor_16',
 'sensor_17',
 'sensor_18',
 'sensor_19',
 'sensor_20',
 'sensor_21',
 'RUL',
 'sensor_2_avg5',
 'sensor_2_avg10',
 'sensor_2_avg20',
 'sensor_2_avg30',
 'sensor_4_avg5',
 'sensor_4_avg10',
 'sensor_4_avg20',
 'sensor_4_avg30',
 'sensor_11_avg5',
 'sensor_11_avg10',
 'sensor_11_avg20',
 'sensor_11_avg30',
 'sensor_15_avg5',
 'sensor_15_avg10',
 'sensor_15_avg20',
 'sensor_15_avg30',
 'sensor_17_avg5',
 'sensor_17_avg10',
 'sensor_17_avg20',
 'sensor_17_avg30',
 'sensor_2_diff',
 'sensor_4_diff',
 'sensor_11_diff',
 'sensor_15_diff',
 'sensor_17_diff']

In [11]:
train_df.shape

(20631, 52)

In [12]:
#Create feature list
features = [

    "cycle",

    "sensor_2",
    "sensor_4",
    "sensor_11",
    "sensor_15",
    "sensor_17",

    "sensor_2_avg5",
    "sensor_4_avg5",
    "sensor_11_avg5",
    "sensor_15_avg5",
    "sensor_17_avg5",

    "sensor_2_avg10",
    "sensor_4_avg10",
    "sensor_11_avg10",
    "sensor_15_avg10",
    "sensor_17_avg10",

    "sensor_2_avg20",
    "sensor_4_avg20",
    "sensor_11_avg20",
    "sensor_15_avg20",
    "sensor_17_avg20",

    "sensor_2_avg30",
    "sensor_4_avg30",
    "sensor_11_avg30",
    "sensor_15_avg30",
    "sensor_17_avg30",

    "sensor_2_diff",
    "sensor_4_diff",
    "sensor_11_diff",
    "sensor_15_diff",
    "sensor_17_diff"
]

In [13]:
X = train_df[features]

y = train_df["RUL"]

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [15]:
from sklearn.ensemble import RandomForestRegressor

rul_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rul_model.fit(
    X_train,
    y_train
)

,n_estimators,300
,criterion,'squared_error'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [16]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

pred = rul_model.predict(X_test)

print(
    "MAE:",
    mean_absolute_error(y_test, pred)
)

print(
    "RMSE:",
    mean_squared_error(y_test, pred) ** 0.5
)

print(
    "R2:",
    r2_score(y_test, pred)
)

MAE: 13.595795368954748
RMSE: 20.34645181850033
R2: 0.9093900929444295


In [17]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rul_model.feature_importances_
})

feature_importance.sort_values(
    by="Importance",
    ascending=False
).head(15)

,Feature,Importance
0,cycle,0.540679
14,sensor_15_avg10,0.112159
11,sensor_2_avg10,0.031426
23,sensor_11_avg30,0.029291
21,sensor_2_avg30,0.029126
22,sensor_4_avg30,0.026201
24,sensor_15_avg30,0.024860
8,sensor_11_avg5,0.021956
25,sensor_17_avg30,0.021590
12,sensor_4_avg10,0.018715


In [18]:
import joblib

joblib.dump(
    rul_model,
    "../models/rul_model.pkl"
)

['../models/rul_model.pkl']

In [19]:
loaded_rul_model = joblib.load(
    "../models/rul_model.pkl"
)

type(loaded_rul_model)

sklearn.ensemble._forest.RandomForestRegressor

In [20]:
hasattr(
    loaded_rul_model,
    "estimators_"
)

True

In [21]:
sample = X_test.iloc[[0]]

loaded_rul_model.predict(sample)

array([133.850128])

In [22]:
joblib.dump(
    X_train.columns.tolist(),
    "../models/rul_features.pkl"
)

['../models/rul_features.pkl']

In [23]:
rul_metrics = {
    "MAE": mean_absolute_error(y_test, pred),
    "RMSE": mean_squared_error(y_test, pred) ** 0.5,
    "R2": r2_score(y_test, pred)
}

joblib.dump(
    rul_metrics,
    "../models/rul_metrics.pkl"
)

['../models/rul_metrics.pkl']

In [24]:
joblib.load(
    "../models/rul_metrics.pkl"
)

{'MAE': 13.595795368954748,
 'RMSE': 20.34645181850033,
 'R2': 0.9093900929444295}

In [25]:
y_train.describe()

count    16504.000000
mean       107.559683
std         69.197147
min          0.000000
25%         51.000000
50%        102.000000
75%        156.000000
max        361.000000
Name: RUL, dtype: float64

In [26]:
y_test.describe()

count    4127.000000
mean      108.800339
std        67.601054
min         0.000000
25%        53.000000
50%       105.000000
75%       155.000000
max       360.000000
Name: RUL, dtype: float64

In [27]:
train_df["RUL"].describe()

count    20631.000000
mean       107.807862
std         68.880990
min          0.000000
25%         51.000000
50%        103.000000
75%        155.000000
max        361.000000
Name: RUL, dtype: float64